In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import auc, roc_curve

In [16]:
df_train = pd.read_csv("./cs-training.csv")
df_test = pd.read_csv("./cs-test.csv")

In [17]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 11  NumberOfDep

In [ ]:
# (Optional) 刪除缺失值
#df_train.dropna(inplace=True)
#df_train.info()
# 直接刪, 分數直接降低


<class 'pandas.DataFrame'>
Index: 120269 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            120269 non-null  int64  
 1   SeriousDlqin2yrs                      120269 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  120269 non-null  float64
 3   age                                   120269 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  120269 non-null  int64  
 5   DebtRatio                             120269 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       120269 non-null  int64  
 8   NumberOfTimes90DaysLate               120269 non-null  int64  
 9   NumberRealEstateLoansOrLines          120269 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  120269 non-null  int64  
 11  NumberOfDependen

In [19]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 101503 entries, 0 to 101502
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            101503 non-null  int64  
 1   SeriousDlqin2yrs                      0 non-null       float64
 2   RevolvingUtilizationOfUnsecuredLines  101503 non-null  float64
 3   age                                   101503 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  101503 non-null  int64  
 5   DebtRatio                             101503 non-null  float64
 6   MonthlyIncome                         81400 non-null   float64
 7   NumberOfOpenCreditLinesAndLoans       101503 non-null  int64  
 8   NumberOfTimes90DaysLate               101503 non-null  int64  
 9   NumberRealEstateLoansOrLines          101503 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  101503 non-null  int64  
 11  NumberOfDep

In [20]:
X = df_train.iloc[:,2:12]
y = df_train["SeriousDlqin2yrs"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [21]:
rf = RandomForestClassifier(random_state=0)

param_grid = {
    'n_estimators' : np.linspace(1, 500, 3, dtype=int),
    'max_depth': [None] + list(np.linspace(1, 500, 2, dtype=int)),
    # 選none 是因為先看 過擬合 對這個資料好不好
    'min_samples_split': np.linspace(2, 100, 2, dtype=int)
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv = 3, # 交叉訓練
    n_jobs=10, # 用幾核心去跑
    verbose=3
)

grid_search.fit(X_train_scaled,y_train)
print("Best Hyperparameters:", grid_search.best_params_)

y_pred = grid_search.predict(X_test_scaled)

auc_roc = roc_auc_score(y_test, y_pred)
print("AUC-ROC:", auc_roc)

Fitting 3 folds for each of 18 candidates, totalling 54 fits
Best Hyperparameters: {'max_depth': None, 'min_samples_split': np.int64(100), 'n_estimators': np.int64(500)}
AUC-ROC: 0.5834324523829519


In [ ]:
# 假設要再弄另一個模型,拿跑出來的參數來跑
#rf= RandomForestClassifier(

#)

In [22]:
X_ = df_test.iloc[:,2:12]

X_scaled = scaler.transform(X_)
y_pred_proba = grid_search.predict_proba(X_scaled)[:,1]

submission = pd.DataFrame({
    'Id': df_test.index + 1,
    'Probability': y_pred_proba
})

submission.to_csv("submission.csv", index=False)